# Semana 2 — Regresión, descenso de gradiente, validación y regularización
### Cuaderno satélite (ALUMNO)

> Completa las celdas marcadas con `# TODO`. El resto está resuelto: ejecútalo y observa.

En esta sesión aprendemos, sobre **ejemplos limpios y controlados**, los fundamentos del aprendizaje supervisado de esta semana:

1. la **regresión lineal** como minimización del error cuadrático medio,
2. el **descenso de gradiente**, que programaremos desde cero,
3. la **ingeniería de características** (hacer lineal lo que no lo era),
4. la **selección de modelo y la validación cruzada** (elegir la complejidad sin autoengañarse),
5. la **regularización** (ridge) como segunda vía para controlar el sobreentrenamiento.

Trabajamos con datos *sintéticos* generados aquí mismo: al conocer la "verdad" que hay detrás, podemos comprobar si cada técnica hace lo que promete. La **transferencia al problema real** —el *soft sensor* de % de sílice sobre `datasets/flotacion_horaria_full.csv`— es la **tarea de la semana** (última sección).

> El cuaderno es largo a propósito y puede repartirse en varias sesiones de clase.

## 0. Preparación

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import pinv
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score

np.set_printoptions(precision=3, suppress=True)

def rmse(a, b):
    """Raíz del error cuadrático medio entre a y b."""
    a = np.asarray(a)
    b = np.asarray(b)
    error = a - b
    mse = np.mean(error**2)
    raiz = np.sqrt(mse)
    return float(raiz)

## 1. Regresión lineal = minimizar el error cuadrático medio

Un modelo lineal predice la salida como combinación de las entradas:

$$\hat{y} = a_0 + a_1 x_1 + a_2 x_2 + \dots = X\,\beta$$

donde a la matriz $X$ le añadimos una columna de unos para el término independiente $a_0$.
"Ajustar" el modelo es elegir $\beta$ que minimice el **error cuadrático medio (MSE)**:

$$J(\beta) = \frac{1}{2n}\sum_{i=1}^{n}\left(y_i - X_i\beta\right)^2$$

Para el caso lineal existe **solución exacta** (la *ecuación normal*), que nos servirá de referencia:

$$\beta^{*} = (X^{\top}X)^{-1}X^{\top}y$$

Generamos un problema lineal de dos variables. **A propósito, las dos entradas tienen escalas muy distintas** (`x1` entre 0 y 1; `x2` entre 0 y 50): esto será clave en la sección 2.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `x1`, `x2` | las dos entradas (120 valores cada una) |
| `y` | la salida que queremos predecir |
| `X` | matriz de diseño: columna de 1s, `x1` y `x2` |
| `beta_ecuacion_normal` | coeficientes calculados con la ecuación normal |

In [ ]:
generador = np.random.default_rng(7)
n_datos = 120
x1 = generador.uniform(0, 1, n_datos)
x2 = generador.uniform(0, 50, n_datos)
ruido = generador.normal(0, 3, n_datos)
# "verdad" oculta:  y = -2 + 5*x1 + 20*x2 + ruido
y = -2 + 5*x1 + 20*x2 + ruido

unos = np.ones(n_datos)   # columna de 1s para a0
X = np.column_stack([unos, x1, x2])
print("Escalas de las entradas:")
print(f"  x1 -> min {x1.min():.2f}  max {x1.max():.2f}")
print(f"  x2 -> min {x2.min():.2f}  max {x2.max():.2f}")

La ecuación normal recupera los coeficientes verdaderos (salvo el ruido):

In [ ]:
beta_ecuacion_normal = pinv(X.T @ X) @ X.T @ y
print("beta (ecuación normal):", beta_ecuacion_normal)
print("verdad oculta:          [-2.  5. 20.]")

## 2. Descenso de gradiente desde cero

La ecuación normal solo existe para el caso lineal y con pocas variables. El **descenso de gradiente** es el método general que sirve para casi todo lo que veremos en el curso (incluidas las redes neuronales de la Semana 5). Parte de un $\beta$ cualquiera y lo corrige repetidamente en la dirección que más reduce el coste:

$$\beta \leftarrow \beta - \alpha\,\nabla J(\beta), \qquad \nabla J(\beta) = \frac{1}{n}X^{\top}(X\beta - y)$$

$\alpha$ es la **tasa de aprendizaje** (cuánto avanzamos en cada paso).

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `descenso_gradiente(X, y, alfa, n_iter)` | devuelve los coeficientes y el historial del MSE |
| `X_normalizada` | `X` con `x1` y `x2` estandarizadas (media 0, desviación 1) |
| `scaler_entradas` | `StandardScaler` ajustado con `x1` y `x2`: guarda su media (`mean_`) y su desviación (`scale_`) |
| `beta_normalizado`, `historial_normalizado` | resultado del descenso con `X_normalizada` |
| `historial_sin_normalizar` | historial del descenso con `X` sin estandarizar (diverge) |
| `beta_ecuacion_normal_normalizado` | coeficientes de la ecuación normal con `X_normalizada`, para comparar |
| `entradas_nuevas`, `entradas_nuevas_normalizadas` | `x1` y `x2` de tres puntos nuevos, antes y después de `scaler_entradas.transform` |
| `y_predicha_nueva` | predicción para los puntos nuevos con `beta_normalizado` |

In [ ]:
def descenso_gradiente(X, y, alfa, n_iter):
    """Descenso de gradiente vectorizado para regresión lineal (MSE).
    Devuelve los coeficientes finales y el historial de coste por iteración."""
    n_datos = X.shape[0]  # número de datos (filas)
    n_coef = X.shape[1]   # número de coeficientes (columnas)
    beta = np.zeros(n_coef)
    historial = np.zeros(n_iter)
    for iteracion in range(n_iter):
        y_estimada = X @ beta
        error = y_estimada - y
        historial[iteracion] = np.mean(error**2)   # MSE actual (ya hecho)

        # === TODO ===
        # 1) gradiente del MSE:   gradiente = (X.T @ error) / n_datos
        # 2) paso de descenso:    beta = beta - alfa * gradiente
        gradiente = (X.T @ error) / n_datos  # <-- TODO
        beta = beta - alfa * gradiente       # <-- TODO

    return beta, historial

### Por qué hay que **normalizar** las entradas

El gradiente usa una **única** tasa $\alpha$ para todas las variables. Si las escalas son muy distintas (como aquí), un $\alpha$ adecuado para `x1` es demasiado grande para `x2` y el método **se dispara**. La solución estándar es **estandarizar** (media 0, desviación 1) antes de entrenar.

Para estandarizar usamos `StandardScaler` de scikit-learn: `fit` calcula la media y la desviación de cada columna, y `transform` aplica $(x - \text{media}) / \text{desviación}$.

Lo comprobamos con la **misma** $\alpha = 0.3$ en ambos casos:

In [ ]:
# Versión estandarizada de las entradas (no tocamos la columna de 1s)
entradas = X[:, 1:]                      # todas las columnas menos la de 1s
scaler_entradas = StandardScaler()
scaler_entradas.fit(entradas)                                # calcula la media y la desviación de x1 y x2
X_normalizada = X.copy()
X_normalizada[:, 1:] = scaler_entradas.transform(entradas)   # (entradas - media) / desviación

beta_normalizado, historial_normalizado = descenso_gradiente(X_normalizada, y, alfa=0.3, n_iter=80)   # normalizado
beta_sin_normalizar, historial_sin_normalizar = descenso_gradiente(X, y, alfa=0.3, n_iter=6)   # SIN normalizar

print("Normalizado: beta final:", beta_normalizado)
print("Sin normalizar: beta final:", beta_sin_normalizar)


La curva de coste del caso normalizado desciende de forma limpia hasta estabilizarse:

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(historial_normalizado, lw=2)
plt.xlabel("iteración")
plt.ylabel("MSE")
plt.title("Descenso de gradiente (entradas normalizadas, α=0.3)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Con las entradas **sin normalizar** y la misma $\alpha = 0.3$, ocurre lo contrario: el MSE se multiplica por unas $10^5$ en cada iteración y el descenso **diverge**. El eje vertical está en escala logarítmica; en escala normal solo se vería el último punto.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.semilogy(historial_sin_normalizar, 'o-', color="tab:red", lw=2)
plt.xlabel("iteración")
plt.ylabel("MSE (escala log)")
plt.title("Descenso de gradiente SIN normalizar (α=0.3): diverge")
plt.grid(alpha=0.3, which="both")
plt.tight_layout()
plt.show()

Y los coeficientes que encuentra coinciden con los de la ecuación normal aplicada a las **mismas entradas normalizadas**. No se parecen a la verdad oculta `[-2, 5, 20]` porque están en la escala de las entradas normalizadas. Para predecir con este modelo, cualquier entrada nueva se normaliza antes con el mismo `scaler_entradas` (`scaler_entradas.transform`); lo hacemos al final de esta sección.

In [ ]:
beta_ecuacion_normal_normalizado = pinv(X_normalizada.T @ X_normalizada) @ X_normalizada.T @ y

print("descenso de gradiente:", beta_normalizado)
print("ecuación normal:      ", beta_ecuacion_normal_normalizado)

y_predicha = X_normalizada @ beta_normalizado
limites = [y.min(), y.max()]
plt.figure(figsize=(5, 4))
plt.plot(y, y_predicha, 'o', alpha=0.6)
plt.plot(limites, limites, 'r--', lw=1)
plt.xlabel("y real")
plt.ylabel("y predicha")
plt.title("Predicción vs realidad")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Aplicar el modelo a datos nuevos

Esta es la forma **real** de usar el modelo: los coeficientes se quedan en la escala normalizada y lo que se normaliza son las **entradas nuevas**. Los pasos son:

1. Normalizar las entradas nuevas con `transform` del **mismo** `scaler_entradas` del entrenamiento. **Nunca** se vuelve a hacer `fit` con los datos nuevos: cambiarían la media y la desviación, y `beta_normalizado` dejaría de corresponder a esas entradas.
2. Añadir la columna de 1s.
3. Multiplicar por `beta_normalizado`.

Para comprobarlo, comparamos con la predicción de la ecuación normal de la sección 1 (`beta_ecuacion_normal`), que usa las entradas originales sin normalizar.

In [ ]:
# Tres puntos nuevos, en las unidades originales
x1_nuevo = np.array([0.2, 0.5, 0.9])
x2_nuevo = np.array([10.0, 25.0, 40.0])
entradas_nuevas = np.column_stack([x1_nuevo, x2_nuevo])

# 1) normalizar con el scaler del entrenamiento (solo transform, sin fit)
entradas_nuevas_normalizadas = scaler_entradas.transform(entradas_nuevas)
# 2) añadir la columna de 1s
unos_nuevos = np.ones(len(x1_nuevo))
X_nueva_normalizada = np.column_stack([unos_nuevos, entradas_nuevas_normalizadas])
# 3) predecir con los coeficientes normalizados
y_predicha_nueva = X_nueva_normalizada @ beta_normalizado

# Comprobación: la ecuación normal con las entradas originales (sin normalizar)
X_nueva = np.column_stack([unos_nuevos, x1_nuevo, x2_nuevo])
y_predicha_ecuacion_normal = X_nueva @ beta_ecuacion_normal

print("entradas normalizadas:")
print(entradas_nuevas_normalizadas)
print("predicción (scaler + beta_normalizado):", y_predicha_nueva)
print("predicción (ecuación normal):          ", y_predicha_ecuacion_normal)

> **Para estirar (opcional).** Prueba la misma función con `alfa` muy grande (p. ej. 1.5) y muy pequeña (p. ej. 0.001): observa cómo en un caso oscila/diverge y en el otro apenas avanza. Añade un criterio de parada por tolerancia (cortar cuando el MSE deje de bajar) en lugar de un número fijo de iteraciones.

## 3. Cuando el lineal no basta: ingeniería de características

Muchos problemas reales **no** son lineales. Pero la regresión lineal es más flexible de lo que parece: es lineal **en los parámetros**, no necesariamente en las variables originales. Si sospechamos qué transformaciones importan (productos, senos, potencias…), las añadimos como **nuevas columnas** y seguimos resolviendo una regresión lineal.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `z1`, `z2` | las dos entradas (70 valores cada una) |
| `y_no_lineal` | la salida, que incluye un producto y un seno |
| `X_plano`, `coef_plano` | matriz de diseño y coeficientes del modelo solo con `z1` y `z2` |
| `X_ampliada`, `coef_ampliada` | lo mismo, añadiendo las columnas `producto` y `seno` |

Generamos un problema deliberadamente no lineal:

In [ ]:
generador = np.random.default_rng(3)
n_datos_no_lineal = 70
z1 = generador.uniform(-2, 2, n_datos_no_lineal)
z2 = generador.uniform(-2, 2, n_datos_no_lineal)
ruido_no_lineal = generador.normal(0, 0.8, n_datos_no_lineal)
# verdad no lineal: incluye un producto y un seno
y_no_lineal = 1 + 2*z1 - 3*z2 + 4*z1*z2 + 1.5*np.sin(3*z1) + ruido_no_lineal

Un modelo lineal **plano** (solo `z1`, `z2`) no puede reproducirlo: R² bajo.

In [ ]:
unos_no_lineal = np.ones(n_datos_no_lineal)
X_plano = np.column_stack([unos_no_lineal, z1, z2])
coef_plano = pinv(X_plano.T @ X_plano) @ X_plano.T @ y_no_lineal
pred_plano = X_plano @ coef_plano
r2_plano = r2_score(y_no_lineal, pred_plano)
print(f"R² (lineal plano) = {r2_plano:.3f}")

limites = [y_no_lineal.min(), y_no_lineal.max()]
plt.figure(figsize=(5, 4))
plt.plot(y_no_lineal, pred_plano, 'o', alpha=0.6)
plt.plot(limites, limites, 'r--', lw=1)
plt.xlabel("y real")
plt.ylabel("y predicha")
plt.title("Modelo lineal plano (insuficiente)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Añadimos las columnas adecuadas (`z1·z2` y `sin(3·z1)`). Sigue siendo una regresión lineal, pero ahora ajusta casi perfectamente:

In [ ]:
# TODO: completa la matriz de diseño añadiendo las columnas z1*z2 y sin(3*z1).
# Pista: calcula primero las dos columnas nuevas y añádelas a la lista de column_stack.
producto = ...                                          # <-- TODO: z1 por z2
seno = ...                                              # <-- TODO: np.sin de 3*z1
X_ampliada = np.column_stack([unos_no_lineal, z1, z2])  # <-- incompleta: añade producto y seno

coef_ampliada = pinv(X_ampliada.T @ X_ampliada) @ X_ampliada.T @ y_no_lineal
pred_ampliada = X_ampliada @ coef_ampliada
r2_ampliada = r2_score(y_no_lineal, pred_ampliada)
print(f"R² (con variables adecuadas) = {r2_ampliada:.3f}")
print("coeficientes:", coef_ampliada)

## 4. Selección de modelo y validación

En la sección 3 sabíamos qué variables añadir. En general **no lo sabemos**, y surge la pregunta central del aprendizaje automático: **¿cómo de complejo debe ser el modelo?**

- Un modelo demasiado simple no captura el patrón (**subentrenamiento**).
- Un modelo demasiado complejo memoriza el ruido y falla con datos nuevos (**sobreentrenamiento**).

La clave: **el error de entrenamiento *siempre* baja al aumentar la complejidad**, así que no sirve para elegir. Hay que medir el error en datos **que el modelo no ha visto** (*validación*).

Usamos el ejemplo canónico: puntos de $\sin(2\pi t)$ con ruido, ajustados con polinomios de orden creciente.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `t_puntos`, `y_puntos` | los 20 datos: posición $t$ y valor medido con ruido |
| `matriz_polinomica(t, orden)` | matriz de diseño con columnas `[1, t, t², …, t^orden]` |
| `idx_train`, `idx_val` | posiciones de los 14 puntos de train y los 6 de validación (partición fija) |
| `ordenes` | órdenes del polinomio que se comparan, de 0 a 11 |
| `kfold` | el reparto en 5 folds |
| `idx_train_ronda`, `idx_val_ronda` | posiciones de train y de validación en cada ronda del $K$-fold |
| `rmse_cv_por_orden` | RMSE medio de las 5 rondas, para cada orden |

In [ ]:
generador = np.random.default_rng(1)
n_puntos = 20
t_puntos = generador.uniform(0, 1, n_puntos)
t_puntos = np.sort(t_puntos)   # de menor a mayor
ruido_puntos = generador.normal(0, 0.25, n_puntos)
y_puntos = np.sin(2*np.pi*t_puntos) + ruido_puntos

def matriz_polinomica(t, orden):
    """Matriz de diseño polinómica: columnas [1, t, t^2, ..., t^orden]."""
    columnas = []
    for potencia in range(orden + 1):
        columnas.append(t**potencia)
    return np.column_stack(columnas)

t_curva = np.linspace(0, 1, 200)
y_verdad = np.sin(2*np.pi*t_curva)
plt.figure(figsize=(6, 3.5))
plt.plot(t_curva, y_verdad, 'g--', lw=1, label="verdad sin(2πt)")
plt.plot(t_puntos, y_puntos, 'or', label="datos con ruido")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Datos de ejemplo")
plt.tight_layout()
plt.show()

**Partición fija (hold-out).** Separamos 70% para entrenar y 30% para validar, y barremos el orden del polinomio. Observa: el error de *train* baja sin parar, pero el de *validación* tiene forma de **U** — el mínimo marca la complejidad adecuada.

In [ ]:
generador_reparto = np.random.default_rng(0)
indices_barajados = generador_reparto.permutation(n_puntos)  # los 20 índices, barajados
idx_train = indices_barajados[:14]                           # los 14 primeros, para train
idx_val = indices_barajados[14:]                             # los 6 restantes, para validación

ordenes = range(0, 12)
rmse_train_por_orden = []
rmse_val_por_orden = []
for orden in ordenes:
    A_train = matriz_polinomica(t_puntos[idx_train], orden)
    coef = pinv(A_train.T @ A_train) @ A_train.T @ y_puntos[idx_train]   # ecuación normal con train
    A_val = matriz_polinomica(t_puntos[idx_val], orden)
    pred_train = A_train @ coef
    pred_val = A_val @ coef
    rmse_train = rmse(pred_train, y_puntos[idx_train])
    rmse_val = rmse(pred_val, y_puntos[idx_val])
    rmse_train_por_orden.append(rmse_train)
    rmse_val_por_orden.append(rmse_val)

plt.figure(figsize=(6, 3.5))
plt.plot(ordenes, rmse_train_por_orden, 'o-', label="train")
plt.plot(ordenes, rmse_val_por_orden, 's-', label="validación")
plt.xlabel("orden del polinomio")
plt.ylabel("RMSE")
plt.ylim(0, 0.8)
plt.legend()
plt.grid(alpha=0.3)
plt.title("Curva de validación (partición fija)")
plt.tight_layout()
plt.show()

posicion = np.argmin(rmse_val_por_orden)   # posición del menor error
print("orden con menor error de validación:", ordenes[posicion])

**Un solo reparto es ruidoso.** Con solo 20 puntos, el orden elegido depende de *qué* 6 puntos caen por azar en validación. Compruébalo: repetimos la partición fija anterior cambiando solo la semilla del reparto.

In [ ]:
# Repetimos el hold-out de la celda anterior con 10 semillas distintas
for semilla in range(10):
    generador = np.random.default_rng(semilla)         # generador de azar con esta semilla
    indices_semilla = generador.permutation(n_puntos)  # los 20 índices, barajados
    idx_train_semilla = indices_semilla[:14]           # los 14 primeros, para train
    idx_val_semilla = indices_semilla[14:]             # los 6 restantes, para validación

    rmse_val_semilla = []
    for orden in ordenes:
        A_train = matriz_polinomica(t_puntos[idx_train_semilla], orden)
        coef = pinv(A_train.T @ A_train) @ A_train.T @ y_puntos[idx_train_semilla]   # ecuación normal con train
        A_val = matriz_polinomica(t_puntos[idx_val_semilla], orden)
        pred_val = A_val @ coef
        rmse_val = rmse(pred_val, y_puntos[idx_val_semilla])
        rmse_val_semilla.append(rmse_val)

    posicion = np.argmin(rmse_val_semilla)   # posición del menor error
    orden_elegido = ordenes[posicion]
    print(f"semilla {semilla}: orden elegido = {orden_elegido}")

Según la semilla, el hold-out elige cualquier orden entre 0 y 7: la decisión depende más del azar que de los datos. Además, en cada reparto hay 6 puntos que nunca se usan para entrenar y 14 que nunca se usan para validar.

### Validación cruzada $K$-fold

La **validación cruzada** ($K$-fold) usa todos los puntos para las dos cosas. Con $N = 20$ puntos y $K = 5$:

1. Se barajan los 20 puntos y se reparten en 5 grupos (*folds*) de 4 puntos.
2. **Ronda 1:** se aparta el fold 1 para validar. Se entrena con los otros 16 puntos y se mide el RMSE en los 4 apartados.
3. **Rondas 2 a 5:** se repite, apartando cada vez un fold distinto. Al final, **cada punto ha validado exactamente una vez** y ha servido para entrenar en las otras cuatro rondas.
4. La puntuación es la **media de los 5 RMSE**.

Cada ronda entrena un modelo **nuevo desde cero**. Esos 5 modelos solo sirven para medir y después se descartan.

En código, `KFold(n_splits=5, shuffle=True, random_state=0)` solo decide el reparto: no ve los datos ni ajusta nada. `kfold.split(t_puntos)` devuelve, ronda a ronda, dos arrays de **posiciones** (índices): `idx_train_ronda`, las de entrenamiento, e `idx_val_ronda`, las de validación. No devuelve los datos: con esos índices seleccionamos `t_puntos[idx_train_ronda]`, `y_puntos[idx_train_ronda]`, etc.. La celda siguiente muestra el reparto.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)

# En cada ronda: qué índices validan y cuántos entrenan
ronda = 1
for idx_train_ronda, idx_val_ronda in kfold.split(t_puntos):
    print(f"ronda {ronda}: validación = {idx_val_ronda}   train = {len(idx_train_ronda)} puntos")
    ronda = ronda + 1



**Ahora usamos el reparto para elegir el orden.** La celda siguiente tiene **dos bucles anidados**:

- El **exterior** recorre los órdenes candidatos (0 a 11).
- El **interior** recorre las 5 rondas. En cada una ajusta el polinomio **solo con `idx_train_ronda`** y mide el RMSE **solo en `idx_val_ronda`**.

Al salir del bucle interior, `rmse_rondas` contiene 5 RMSE (uno por ronda) y su media es la puntuación de ese orden. Al final, `rmse_cv_por_orden` tiene una puntuación por orden y elegimos la menor. En total se ajustan 12 × 5 = 60 modelos.

Como `random_state=0` está fijo, cada llamada a `kfold.split(t_puntos)` repite el mismo reparto: todos los órdenes se comparan con los mismos folds, y la comparación es justa.

In [ ]:
# kfold es el KFold de la celda anterior (mismo reparto para todos los órdenes)
rmse_cv_por_orden = []   # una puntuación por orden: la media de sus 5 RMSE
for orden in ordenes:   # bucle exterior: cada orden candidato
    rmse_rondas = []   # aquí se guardan los 5 RMSE de este orden
    for idx_train_ronda, idx_val_ronda in kfold.split(t_puntos):   # bucle interior: las 5 rondas
        # === TODO ===
        # Ajusta un polinomio de grado `orden` con los puntos de TRAIN (idx_train_ronda) y
        # calcula el RMSE en los de VALIDACIÓN (idx_val_ronda). Sigue el patrón del hold-out.
        A_train = ...   # <-- TODO: matriz de diseño con train:   matriz_polinomica(t_puntos[idx_train_ronda], orden)
        coef = ...      # <-- TODO: ecuación normal:   pinv(A_train.T @ A_train) @ A_train.T @ y_puntos[idx_train_ronda]
        A_val = ...     # <-- TODO: matriz de diseño con validación:   matriz_polinomica(t_puntos[idx_val_ronda], orden)
        pred_val = ...  # <-- TODO: A_val @ coef
        rmse_val = ...  # <-- TODO: RMSE en validación:   rmse(pred_val, y_puntos[idx_val_ronda])
        rmse_rondas.append(rmse_val)
    rmse_medio = np.mean(rmse_rondas)   # media de las 5 rondas = puntuación de este orden
    rmse_cv_por_orden.append(rmse_medio)

posicion = np.argmin(rmse_cv_por_orden)
mejor_orden = ordenes[posicion]

plt.figure(figsize=(6, 3.5))
plt.plot(ordenes, rmse_cv_por_orden, 's-', color="C1")
plt.axvline(mejor_orden, color="k", ls=":", lw=1)
plt.xlabel("orden del polinomio")
plt.ylabel("RMSE validación (5-fold)")
plt.ylim(0, 0.8)
plt.grid(alpha=0.3)
plt.title(f"Validación cruzada → mejor orden = {mejor_orden}")
plt.tight_layout()
plt.show()
print("mejor orden por validación cruzada:", mejor_orden)

La validación cruzada elige un orden **bajo y robusto** (un polinomio de grado 3–4 basta para una onda) y castiga con dureza los órdenes altos, que en algún fold se disparan. Esta es la herramienta honesta para fijar la complejidad.

Dos matices:

- Los modelos de cada ronda solo sirven para **puntuar** un orden. Una vez elegido el orden, el modelo definitivo se ajusta con todos los datos de entrenamiento.
- `shuffle=True` baraja los puntos antes de repartirlos. Aquí es correcto porque los puntos son independientes entre sí. En una serie temporal como la de la tarea no lo es (véase la sección 6).

> **Para estirar (opcional).** Repite la validación cruzada con `random_state` = 1, 2, …, 9 en `KFold` y compara con las 10 semillas del hold-out. El orden elegido por 5-fold cambia mucho menos.

## 5. Regularización

Reducir el orden es una forma de controlar la complejidad. La **regularización** es otra: dejamos un modelo flexible (orden alto) pero **penalizamos los coeficientes grandes**, que son la firma del sobreentrenamiento.

- **Ridge (L2)**: $J + \lambda\sum \beta_j^2$ — encoge los coeficientes.

$\lambda$ controla la intensidad, y se elige **igual que el orden: por validación**. Requisito imprescindible: **estandarizar** las variables antes.

Variables de esta sección (los datos e `idx_train`/`idx_val` son los de la sección 4):

| Variable | Qué contiene |
|---|---|
| `orden_alto` | 14, un orden tan alto que provoca sobreentrenamiento |
| `diseno_sin_unos(t, orden)` | matriz de diseño con columnas `[t, t², …, t^orden]` (el modelo añade el término independiente) |
| `X_train`, `X_val` | matrices de diseño estandarizadas de la partición fija |
| `ols` | regresión lineal sin regularizar |
| `lambdas`, `lambda_optimo` | valores de $\lambda$ probados y el elegido por validación cruzada |
| `ridge_optimo` | Ridge ajustado con `lambda_optimo` |

Partimos de un polinomio deliberadamente alto (**orden 14**, tantos términos como puntos de train) que se sobreentrena de forma extrema:

In [ ]:
orden_alto = 14

def diseno_sin_unos(t, orden):
    """Columnas [t, t^2, ..., t^orden], sin la de 1s: el término independiente lo pone el modelo."""
    columnas = []
    for potencia in range(1, orden + 1):
        columnas.append(t**potencia)
    return np.column_stack(columnas)

D_train = diseno_sin_unos(t_puntos[idx_train], orden_alto)
D_val = diseno_sin_unos(t_puntos[idx_val], orden_alto)
scaler = StandardScaler()
scaler.fit(D_train)   # ajustar SOLO con train
X_train = scaler.transform(D_train)
X_val = scaler.transform(D_val)

ols = LinearRegression()
ols.fit(X_train, y_puntos[idx_train])
pred_train = ols.predict(X_train)
pred_val = ols.predict(X_val)
rmse_train = rmse(pred_train, y_puntos[idx_train])
rmse_val = rmse(pred_val, y_puntos[idx_val])
print(f"OLS orden 14 -> RMSE train = {rmse_train:.3f}   RMSE val = {rmse_val:.1f}")
print("El error de train queda por debajo del ruido (0,25), pero el de validación es enorme: sobreentrenamiento de manual.")

**Elegimos $\lambda$ por validación cruzada**, igual que elegimos el orden en la sección 4. El error de train baja al reducir $\lambda$, pero el de validación tiene forma de **U**: su mínimo da el $\lambda$ adecuado. (Eje vertical en escala log para que quepan ambos.)

Los dos bucles son los de la sección 4, con los mismos folds (`kfold`): el exterior recorre los valores de $\lambda$ y el interior las 5 rondas. Hay un detalle nuevo: en cada ronda, el `StandardScaler` se ajusta **solo con los puntos de train de esa ronda** (`idx_train_ronda`). Si lo ajustáramos con los 20 puntos, la media y la desviación de los puntos de validación se colarían en el entrenamiento (fuga). La curva *train (CV)* es la media del error de train de las 5 rondas.

In [ ]:
lambdas = np.logspace(-4, 2, 25)
rmse_cv_train_por_lambda = []  # error medio de train de las 5 rondas, por λ
rmse_cv_val_por_lambda = []    # error medio de validación de las 5 rondas, por λ
for lam in lambdas:            # bucle exterior: cada λ candidato
    rmse_train_rondas = []
    rmse_val_rondas = []
    for idx_train_ronda, idx_val_ronda in kfold.split(t_puntos):   # bucle interior: las 5 rondas
        D_train_ronda = diseno_sin_unos(t_puntos[idx_train_ronda], orden_alto)
        D_val_ronda = diseno_sin_unos(t_puntos[idx_val_ronda], orden_alto)
        scaler_ronda = StandardScaler()
        scaler_ronda.fit(D_train_ronda)   # escalado ajustado solo con el train de la ronda
        X_train_ronda = scaler_ronda.transform(D_train_ronda)
        X_val_ronda = scaler_ronda.transform(D_val_ronda)
        ridge_ronda = Ridge(alpha=lam)
        ridge_ronda.fit(X_train_ronda, y_puntos[idx_train_ronda])
        pred_train_ronda = ridge_ronda.predict(X_train_ronda)
        pred_val_ronda = ridge_ronda.predict(X_val_ronda)
        rmse_train = rmse(pred_train_ronda, y_puntos[idx_train_ronda])
        rmse_val = rmse(pred_val_ronda, y_puntos[idx_val_ronda])
        rmse_train_rondas.append(rmse_train)
        rmse_val_rondas.append(rmse_val)
    rmse_train_medio = np.mean(rmse_train_rondas)
    rmse_val_medio = np.mean(rmse_val_rondas)
    rmse_cv_train_por_lambda.append(rmse_train_medio)
    rmse_cv_val_por_lambda.append(rmse_val_medio)

posicion = np.argmin(rmse_cv_val_por_lambda)
lambda_optimo = lambdas[posicion]
rmse_cv_optimo = rmse_cv_val_por_lambda[posicion]

plt.figure(figsize=(6, 3.5))
plt.loglog(lambdas, rmse_cv_train_por_lambda, 'o-', ms=3, label="train (CV)")
plt.loglog(lambdas, rmse_cv_val_por_lambda, 's-', ms=3, label="validación (CV)")
plt.axvline(lambda_optimo, color="k", ls=":", lw=1)
plt.xlabel("λ (escala log)")
plt.ylabel("RMSE (escala log)")
plt.legend()
plt.grid(alpha=0.3, which="both")
plt.title(f"Ridge: λ óptimo por CV ≈ {lambda_optimo:.3f}")
plt.tight_layout()
plt.show()
print(f"λ óptimo por validación cruzada ≈ {lambda_optimo:.3f}   RMSE val (CV) = {rmse_cv_optimo:.3f}")

El $\lambda$ elegido **doma** el polinomio de orden 14: recupera un ajuste suave, comparable al mejor orden de la sección 4. El OLS sin regularizar oscila tan violentamente que **se sale del gráfico**.

In [ ]:
ridge_optimo = Ridge(alpha=lambda_optimo)
ridge_optimo.fit(X_train, y_puntos[idx_train])

t_curva = np.linspace(0, 1, 200)
D_curva = diseno_sin_unos(t_curva, orden_alto)
X_curva = scaler.transform(D_curva)
y_verdad = np.sin(2*np.pi*t_curva)
pred_ols_curva = ols.predict(X_curva)
pred_ridge_curva = ridge_optimo.predict(X_curva)

plt.figure(figsize=(6, 3.5))
plt.plot(t_curva, y_verdad, 'g--', lw=1, label="verdad")
plt.plot(t_puntos[idx_train], y_puntos[idx_train], 'ok', ms=4, label="train")
plt.plot(t_curva, pred_ols_curva, 'r-', lw=1, alpha=0.7, label="OLS orden 14 (se dispara)")
plt.plot(t_curva, pred_ridge_curva, 'b-', lw=2, label=f"Ridge λ≈{lambda_optimo:.3f}")
plt.ylim(-2, 2)
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.title("Regularización frente a sobreentrenamiento")
plt.tight_layout()
plt.show()

Para terminar, comparamos los dos modelos en la partición fija: el error de validación y el **mayor coeficiente en valor absoluto**. Ridge conserva los 14 términos, pero los **encoge**: sus coeficientes son mucho menores que los del OLS.

In [ ]:
def mayor_coeficiente(coef):
    """Mayor coeficiente en valor absoluto."""
    absolutos = np.abs(coef)
    return absolutos.max()

print(f"{'modelo':18s} {'RMSE_val':>9s} {'mayor |coef|':>13s}")
print("-" * 42)

# Ejemplo ya hecho (OLS):
pred_val = ols.predict(X_val)
rmse_val = rmse(pred_val, y_puntos[idx_val])
mayor = mayor_coeficiente(ols.coef_)
nombre = "OLS orden 14"
print(f"{nombre:18s} {rmse_val:9.1f} {mayor:13.1f}")

# === TODO ===
# Añade la fila de Ridge con el λ elegido por validación cruzada (usa ridge_optimo).
# Sigue el patrón del OLS, una operación por línea: predice en X_val, calcula el RMSE
# con y_puntos[idx_val] y el mayor coeficiente con mayor_coeficiente(ridge_optimo.coef_).
# Como nombre, usa f"Ridge λ≈{lambda_optimo:.3f}" y escribe los números con 3 decimales.

> **La regla honesta.** Tienes **dos mandos** para el sobreentrenamiento: la **complejidad** (orden / número de variables) y la **regularización** ($\lambda$). En ambos casos **quien decide es la validación**, nunca el error de entrenamiento. Ridge estabiliza los coeficientes encogiéndolos. La regularización no es un botón de "mejorar": solo cobra cuando hay riesgo real de sobreentrenamiento (con pocas variables limpias, no cambiaría nada).

## 6. Puente a la tarea de la semana (proyecto ancla)

Ya tienes las piezas. La **tarea** consiste en llevarlas al problema real: construir el *soft sensor* que estima el **% de sílice del concentrado** a partir de las variables de proceso, sobre `datasets/flotacion_horaria_full.csv` (la tabla horaria de los ~6 meses, construida con la lógica de la Semana 1).

Seis recordatorios que separan un modelo honesto de uno tramposo:

1. **Disciplina anti-fuga.** Como características, **solo variables de proceso**. **Nunca** las columnas de concentrado: *son* la respuesta.
2. **Partición temporal, no aleatoria.** Entrena con las primeras horas y evalúa con las últimas. Aquí la validación cruzada aleatoria de la sección 4 **no** es válida tal cual: mezclaría horas contiguas casi idénticas. Se usa validación **por bloques temporales**.
3. **Baseline primero.** Comprueba que tu modelo **bate a predecir la media**. Si no la bate, no vale.
4. **Elige la complejidad por validación.** ¿Cuántas variables, qué transformaciones, qué $\lambda$? Lo decide el error de validación (temporal), no el de entrenamiento.
5. **Estandariza** antes de aplicar ridge.
6. **Compara con la persistencia** ("la sílice de ahora ≈ la de hace una hora"). Que un lineal estático no la supere es un **resultado**, no un fracaso: es la puerta a las redes secuenciales de la Semana 5.

El enunciado completo (con niveles mínimo y de ampliación) y el guion de la defensa oral van aparte.